# Lab 11: Grid Localization using Bayes Filter (Virtual Robot)

### <span style="color:rgb(0,150,0)">It is recommended that you close any heavy-duty applications running on your system while working on this lab.</span>

<hr>


In [1]:
%load_ext autoreload
%autoreload 2

import traceback
from notebook_utils import *
from Traj import *
import asyncio
from localization_extras import Localization

# Setup Logger
LOG = get_logger('demo_notebook.log')

# Init GUI and Commander
gui = GET_GUI()
cmdr = gui.launcher.commander

gui.show()

# Start the simulator
START_SIM()

# Start the plotter
START_PLOTTER()

/home/leslier/programming/ece5160-fast-robotics/FastRobots_ble/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


2026-04-22 14:05:34,299 | INFO     |: Logger demo_notebook.log initialized.


TwoByTwoLayout(children=(Label(value='Simulator', layout=Layout(grid_area='top-left', width='80px')), HBox(chi…

Loading Flatland...
Initializing pygame framework...


In [2]:
# Initialize Robot to communicate with the virtual robot and plotter
robot = VirtualRobot(cmdr)

# Initialize mapper
# Requires a VirtualRobot object as a parameter
mapper = Mapper(robot)

# Initialize your BaseLocalization object
# Requires a VirtualRobot object and a Mapper object as parameters
loc = Localization(robot, mapper)

## Plot Map
cmdr.plot_map()

2026-04-22 14:05:41,289 | INFO     |:  | Number of observations per grid cell: 18
2026-04-22 14:05:41,291 | INFO     |:  | Precaching Views...


/home/leslier/programming/ece5160-fast-robotics/jupyter_notebooks/FastRobots-sim-release/localization.py:150: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(distance_intersections_tt), intersections_tt[np.nanargmin(distance_intersections_tt)]


2026-04-22 14:05:43,003 | INFO     |:  | Precaching Time: 1.711 secs
2026-04-22 14:05:43,005 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-04-22 14:05:43,005 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107


# Run the Bayes Filter
The cells below utilizes the member functions of class **Localization** (defined in [localization_extras.py](../localization_extras.py)) in each iteration of the Bayes filter algorithm to localize the robot in the grid map. <br>

In [3]:
# Reset Robot and Plots
robot.reset()
cmdr.reset_plotter()

# Init Uniform Belief
loc.init_grid_beliefs()

# Get Observation Data by executing a 360 degree rotation motion
loc.get_observation_data()

# Run Update Step
loc.update_step()
loc.print_update_stats(plot_data=True)

# Plot Odom and GT
current_odom, current_gt = robot.get_pose()
cmdr.plot_gt(current_gt[0], current_gt[1])
cmdr.plot_odom(current_odom[0], current_odom[1])

2026-04-22 14:05:53,646 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-04-22 14:05:53,647 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107
2026-04-22 14:05:56,499 | INFO     |: Update Step
2026-04-22 14:05:56,503 | INFO     |:      | Update Time: 0.002 secs
2026-04-22 14:05:56,504 | INFO     |: ---------- UPDATE STATS -----------
2026-04-22 14:05:56,509 | INFO     |: GT index      : (6, 4, 9)
2026-04-22 14:05:56,510 | INFO     |: Bel index     : (np.int64(5), np.int64(4), np.int64(9)) with prob = 0.9001118
2026-04-22 14:05:56,511 | INFO     |: Bel_bar prob at index = 0.00051440329218107
2026-04-22 14:05:56,512 | INFO     |: GT            : (0.000, 0.000, 360.000)
2026-04-22 14:05:56,513 | INFO     |: Belief        : (0.000, 0.000, 10.000)
2026-04-22 14:05:56,514 | INFO     |: POS ERROR     : (-0.000, -0.000, 350.000)
2026-04-22 14:05:56,515 | INFO     |: ---------- UPDATE STATS -----------


In [4]:
# Initialize the Trajectory object
traj = Trajectory(loc)

# Run through each motion steps
for t in range(0, traj.total_time_steps):
    print("\n\n-----------------", t, "-----------------")
    
    prev_odom, current_odom, prev_gt, current_gt = traj.execute_time_step(t)
        
    # Prediction Step
    loc.prediction_step(current_odom, prev_odom)
    loc.print_prediction_stats(plot_data=True)
    
    # Get Observation Data by executing a 360 degree rotation motion
    loc.get_observation_data()
    
    # Update Step
    loc.update_step()
    loc.print_update_stats(plot_data=True)

# Uncomment the below line to wait for keyboard input between each iteration.
#   input("Press Enter to Continue")
        
    print("-------------------------------------")



----------------- 0 -----------------
2026-04-22 14:06:04,338 | INFO     |: Prediction Step
2026-04-22 14:06:04,393 | INFO     |:  | Prediction Time: 0.054 secs
2026-04-22 14:06:04,394 | INFO     |: ---------- PREDICTION STATS -----------
2026-04-22 14:06:04,401 | INFO     |: GT index         : (6, 3, 6)
2026-04-22 14:06:04,402 | INFO     |: Prior Bel index  : (np.int64(7), np.int64(1), np.int64(7)) with prob = 0.0714731
2026-04-22 14:06:04,403 | INFO     |: POS ERROR        : (-0.319, 0.821, -11.062)
2026-04-22 14:06:04,404 | INFO     |: ---------- PREDICTION STATS -----------
2026-04-22 14:06:07,265 | INFO     |: Update Step
2026-04-22 14:06:07,268 | INFO     |:      | Update Time: 0.001 secs
2026-04-22 14:06:07,268 | INFO     |: ---------- UPDATE STATS -----------
2026-04-22 14:06:07,278 | INFO     |: GT index      : (6, 3, 6)
2026-04-22 14:06:07,279 | INFO     |: Bel index     : (np.int64(6), np.int64(4), np.int64(6)) with prob = 1.0
2026-04-22 14:06:07,279 | INFO     |: Bel_bar 